In [27]:
import sqlite3
import pandas as pd


conn = sqlite3.connect("marketing.db")
cleaned_journeys = pd.read_csv("cleaned_journey.csv")
conv = pd.read_csv("conversions.csv")
spend = pd.read_csv("marketing_spend.csv")

In [28]:
cleaned_journeys.to_sql(
    "marketing",
    conn,
    if_exists="replace",
    index=False
)

conv.to_sql(
    "conversions",
    conn,
    if_exists="replace",
    index=False
)

spend.to_sql(
    "marketing_spend",
    conn,
    if_exists="replace",
    index=False
)

540

In [26]:
query = """
SELECT *
FROM marketing
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,user_id,touch_date,channel,campaign_id,session_id,utm_source,month,day
0,U1000,2025-01-30,Facebook,F001,S11015,unknown,1,Thursday
1,U1000,2025-01-31,Google Ads,G002,S18415,Google Ads,1,Friday
2,U1000,2025-02-01,LinkedIn,L001,S72292,LinkedIn,2,Saturday
3,U1000,2025-02-02,Facebook,F001,S16970,Facebook,2,Sunday
4,U1000,2025-02-03,LinkedIn,L001,S30384,LinkedIn,2,Monday
5,U1001,2025-01-01,Google Ads,G002,S22763,unknown,1,Wednesday
6,U1001,2025-01-02,Google Ads,G001,S90642,Google Ads,1,Thursday
7,U1001,2025-01-03,Google Ads,G001,S20916,Google Ads,1,Friday
8,U1002,2025-03-15,LinkedIn,L001,S66397,LinkedIn,3,Saturday
9,U1002,2025-03-16,Google Ads,G002,S63932,Google Ads,3,Sunday


In [29]:
query = """
SELECT COUNT(*) as total_rows
FROM marketing;
"""

pd.read_sql_query(query, conn)

,total_rows
0,3549


In [ ]:
#create a new table called journey_master that combines the marketing and conversions tables
query = """
CREATE TABLE journey_master AS

SELECT
    j.user_id,
    j.touch_date,
    j.channel,
    j.campaign_id,
    c.conversion_date,
    c.revenue

FROM marketing j

LEFT JOIN conversions c
ON j.user_id = c.user_id;
"""

conn.execute(query)
conn.commit()

In [ ]:
#selecting a sample of rows from the journey_master table
query = """
SELECT *
FROM journey_master
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,user_id,touch_date,channel,campaign_id,conversion_date,revenue
0,U1000,2025-01-30,Facebook,F001,2025-03-09,412.0
1,U1000,2025-01-31,Google Ads,G002,2025-03-09,412.0
2,U1000,2025-02-01,LinkedIn,L001,2025-03-09,412.0
3,U1000,2025-02-02,Facebook,F001,2025-03-09,412.0
4,U1000,2025-02-03,LinkedIn,L001,2025-03-09,412.0
5,U1001,2025-01-01,Google Ads,G002,NaN,NaN
6,U1001,2025-01-02,Google Ads,G001,NaN,NaN
7,U1001,2025-01-03,Google Ads,G001,NaN,NaN
8,U1002,2025-03-15,LinkedIn,L001,2025-03-21,746.0
9,U1002,2025-03-16,Google Ads,G002,2025-03-21,746.0


In [ ]:
#selecting the touch order for each user based on touch_date
query = """
SELECT
    user_id,
    touch_date,
    channel,

    ROW_NUMBER() OVER(
        PARTITION BY user_id
        ORDER BY touch_date
    ) AS touch_order

FROM journey_master;
"""

pd.read_sql_query(query, conn)

,user_id,touch_date,channel,touch_order
0,U1000,2025-01-30,Facebook,1
1,U1000,2025-01-31,Google Ads,2
2,U1000,2025-02-01,LinkedIn,3
3,U1000,2025-02-02,Facebook,4
4,U1000,2025-02-03,LinkedIn,5
...,...,...,...,...
3544,U1998,2025-03-20,LinkedIn,5
3545,U1999,2025-02-08,TikTok,1
3546,U1999,2025-02-09,Facebook,2
3547,U1999,2025-02-10,Facebook,3


In [33]:
#verify the touch order for a specific user
query = """
SELECT
    user_id,
    touch_date,
    channel,

    ROW_NUMBER() OVER(
        PARTITION BY user_id
        ORDER BY touch_date
    ) AS touch_order

FROM journey_master

WHERE user_id='U1000';
"""

pd.read_sql_query(query, conn)

,user_id,touch_date,channel,touch_order
0,U1000,2025-01-30,Facebook,1
1,U1000,2025-01-31,Google Ads,2
2,U1000,2025-02-01,LinkedIn,3
3,U1000,2025-02-02,Facebook,4
4,U1000,2025-02-03,LinkedIn,5


In [38]:
pd.read_sql_query("""
SELECT *
FROM first_touch
LIMIT 10;
""", conn)

,user_id,channel,revenue,rn
0,U1000,Facebook,412.0,1
1,U1001,Google Ads,NaN,1
2,U1002,LinkedIn,746.0,1
3,U1003,Facebook,850.0,1
4,U1004,LinkedIn,NaN,1
5,U1005,LinkedIn,222.0,1
6,U1006,Google Ads,NaN,1
7,U1007,LinkedIn,NaN,1
8,U1008,Facebook,NaN,1
9,U1009,Facebook,NaN,1


In [ ]:
# Reading the first_touch table and calculating revenue by channel in descending order
query = """
SELECT
    channel,
    SUM(revenue) AS revenue
FROM first_touch
GROUP BY channel
ORDER BY revenue DESC;
"""

pd.read_sql_query(query, conn)

,channel,revenue
0,Facebook,34856
1,LinkedIn,34624
2,Google Ads,29460
3,TikTok,28292


In [ ]:
# Selecting a 20-sample of rows from the linear_attribution table
query = """
SELECT *
FROM linear_attribution
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,user_id,channel,revenue,total_touches,attribution_credit
0,U1000,Facebook,412.0,5,82.400000
1,U1000,Google Ads,412.0,5,82.400000
2,U1000,LinkedIn,412.0,5,82.400000
3,U1000,Facebook,412.0,5,82.400000
4,U1000,LinkedIn,412.0,5,82.400000
5,U1001,Google Ads,NaN,3,NaN
6,U1001,Google Ads,NaN,3,NaN
7,U1001,Google Ads,NaN,3,NaN
8,U1002,LinkedIn,746.0,3,248.666667
9,U1002,Google Ads,746.0,3,248.666667


In [ ]:
# Reading the linear_attribution table and calculating revenue by channel
query = """
SELECT
    channel,
    ROUND(
        SUM(attribution_credit),
        2
    ) AS revenue

FROM linear_attribution

GROUP BY channel

ORDER BY revenue DESC;
"""

pd.read_sql_query(query, conn)

,channel,revenue
0,LinkedIn,32628.73
1,Google Ads,32250.45
2,TikTok,31228.38
3,Facebook,31124.43


In [47]:
# Reading the first_touch table and linear_attribution table 
first_touch = pd.read_sql_query(
    "SELECT * FROM first_touch",
    conn
)

linear = pd.read_sql_query(
    "SELECT * FROM linear_attribution",
    conn
)

In [ ]:
# Saving the first_touch and linear_attribution tables as CSV files
first_touch.to_csv(
    "first_touch.csv",
    index=False
)

linear.to_csv(
    "linear_attribution.csv",
    index=False
)